# F02 CASTELLAN — HUD de Contrôle & Simulation
## PENTERACT DORN V3 — VIIe Légion

**Rôle** : Simuler l'animation, éditer le `plan_de_vol.json`, figer pour F03.

**Avant de lancer :**
1. Copier `F01_POLUX/OUT/plan_de_vol.json` → `F02_CASTELLAN/IN/plan_de_vol.json`
2. Copier `F01_POLUX/OUT/images/` → `F02_CASTELLAN/IN/images/`
3. Lancer toutes les cellules dans l'ordre
4. Ouvrir l'URL Streamlit générée

In [ ]:
# CELLULE 1 — Montage Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive monté.')

In [ ]:
# CELLULE 2 — Configuration + Téléchargement depuis GitHub (source de vérité)
import shutil, os, subprocess

DRIVE_BASE = '/content/drive/MyDrive/DRIVE_DORN'  # ← adapter si besoin
LOCAL_BASE = '/tmp/DRIVE_DORN'
GITHUB_RAW = 'https://raw.githubusercontent.com/kioka8877-ux/DORN/main'

# Structure locale
os.makedirs(LOCAL_BASE, exist_ok=True)

# Copie Drive → local pour les assets (images, plan_de_vol IN, etc.)
if not os.path.exists(os.path.join(LOCAL_BASE, 'F02_CASTELLAN')):
    print('Copie DRIVE_DORN vers /tmp...')
    shutil.copytree(DRIVE_BASE, LOCAL_BASE, dirs_exist_ok=True)
    print(f'Copie terminée en local : {LOCAL_BASE}')
else:
    print(f'Cache local déjà présent : {LOCAL_BASE}')

# Télécharge les scripts depuis GitHub — TOUJOURS, écrase la version locale
scripts = [
    ('F02_CASTELLAN/CODEBASE/drn_f02_castellan.py',
     f'{LOCAL_BASE}/F02_CASTELLAN/CODEBASE/drn_f02_castellan.py'),
    ('CRS_CUSTOS.py', f'{LOCAL_BASE}/CRS_CUSTOS.py'),
]
for gh_path, dst in scripts:
    url = f'{GITHUB_RAW}/{gh_path}'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    r = subprocess.run(['wget', '-q', '-O', dst, url], capture_output=True)
    status = 'OK' if r.returncode == 0 else 'ERREUR'
    print(f'GitHub → {gh_path} : {status}')

print(f'\nDrive base  : {DRIVE_BASE}')
print(f'Local base  : {LOCAL_BASE}')

In [ ]:
# CELLULE 3 — Installation des dépendances
import subprocess
subprocess.run(['pip', 'install', 'streamlit', '-q'], check=True)
print('Streamlit prêt.')

In [ ]:
# CELLULE 4 — Copie du script depuis LOCAL_BASE (version GitHub téléchargée en C2)
# FIX: utilise LOCAL_BASE (contient la version GitHub) et non DRIVE_BASE (ancienne version Drive)
import shutil, os
script_src = os.path.join(LOCAL_BASE, 'F02_CASTELLAN', 'CODEBASE', 'drn_f02_castellan.py')
script_dst = '/content/drn_f02_castellan.py'
shutil.copy2(script_src, script_dst)
print(f'Script copié depuis LOCAL_BASE : {script_dst}')
# Vérification : affiche la première ligne pour confirmer la version
with open(script_dst) as f:
    print('Version :', f.readline().strip())

In [ ]:
# CELLULE 5 — Lancement Streamlit en arrière-plan
import subprocess, time

# Tuer tout process streamlit résiduel
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(2)

proc = subprocess.Popen(
    ['streamlit', 'run', script_dst,
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false',
     '--', '--drive-base', LOCAL_BASE],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(6)
print('Streamlit lancé (PID:', proc.pid, ')')

In [ ]:
# CELLULE 6 — URL d'accès (port forwarding Colab)
from google.colab.output import eval_js
url = eval_js('google.colab.kernel.proxyPort(8501)')
print(f'\n╔══════════════════════════════════════════╗')
print(f'║  F02 CASTELLAN — URL Streamlit           ║')
print(f'║  {url:<40}║')
print(f'╚══════════════════════════════════════════╝')
print('\nOuvrir ce lien dans un nouvel onglet.')

In [ ]:
# CELLULE 7 — Transit CRS_CUSTOS (check-in F02)
# À lancer UNIQUEMENT après avoir cliqué FIGER LE PLAN DE VOL dans Streamlit
import shutil, subprocess, sys, os
custos_src = os.path.join(LOCAL_BASE, 'CRS_CUSTOS.py')
shutil.copy2(custos_src, '/content/CRS_CUSTOS.py')
result = subprocess.run(
    [sys.executable, '/content/CRS_CUSTOS.py',
     '--frigate', 'F02', '--mode', 'check-in', '--drive-base', LOCAL_BASE],
    capture_output=False
)
if result.returncode == 0:
    print('\nCRS_CUSTOS — F02 check-in OK — Transit autorisé vers F03')
else:
    print('\nCRS_CUSTOS — F02 check-in FAIL')